In [1]:
import os
import requests
import zipfile
import shutil
import re
import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
DATA_URL = 'https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/x8psbz3f6x-2.zip'
ZIP_NAME = 'dataset.zip'

BASE_DIR = 'datapoints'
RAW_DIR = os.path.join(BASE_DIR, 'raw')
META_PATH = os.path.join(BASE_DIR, 'metadata.xlsx')


In [2]:

# -----------------------------
# LABEL MAPPING (unchanged)
# -----------------------------
label_map = {
    '1': 'BEO',
    '2': 'CLH',
    '3': 'CRH',
    '4': 'DLF',
    '5': 'PLF',
    '6': 'DRF',
    '7': 'PRF',
    '8': 'Rest',
}

label_encoding = {
    'BEO': 0,
    'CLH': 1,
    'CRH': 2,
    'DLF': 3,
    'PLF': 4,
    'DRF': 5,
    'PRF': 6,
    'Rest': 7,
}


In [3]:

# -----------------------------
# STEP 1: DOWNLOAD ZIP
# -----------------------------
print('Downloading dataset...')
with requests.get(DATA_URL, stream=True) as r:
    r.raise_for_status()
    with open(ZIP_NAME, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
print('Download complete.')

# -----------------------------
# STEP 2: CREATE DIRECTORIES
# -----------------------------
os.makedirs(RAW_DIR, exist_ok=True)

# -----------------------------
# STEP 3: EXTRACT S* FOLDERS
# -----------------------------
print('Extracting dataset...')
with zipfile.ZipFile(ZIP_NAME, 'r') as zip_ref:
    all_items = zip_ref.namelist()
    root_folder = os.path.commonprefix(all_items).split('/')[0]

    for item in all_items:
        # only extract subject folders S1, S2, ...
        if re.match(rf'{root_folder}/S\d+/', item):
            rel_path = os.path.relpath(item, root_folder)
            target_path = os.path.join(RAW_DIR, rel_path)

            if item.endswith('/'):
                os.makedirs(target_path, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                with zip_ref.open(item) as src, open(target_path, 'wb') as dst:
                    shutil.copyfileobj(src, dst)

print('Extraction complete.')


Download complete.
Extracting dataset...
Extraction complete.


In [ ]:
import openpyxl
# -----------------------------
# STEP 4: METADATA GENERATION
# -----------------------------
rows = []
srno = 1

def patient_sort_key(folder):
    match = re.match(r'S(\d+)', folder)
    return int(match.group(1)) if match else float('inf')

for patient_folder in sorted(os.listdir(RAW_DIR), key=patient_sort_key):
    if not patient_folder.startswith('S'):
        continue

    match_patient = re.match(r'S(\d+)', patient_folder)
    if not match_patient:
        continue

    patient_num = int(match_patient.group(1))
    patient_path = os.path.join(RAW_DIR, patient_folder)

    for file in sorted(os.listdir(patient_path)):
        if not file.endswith('.csv'):
            continue

        base = file[:-4]  # remove .csv
        # Example: S24R1I6_5
        match = re.match(r'S(\d+)R(\d+)([MI])(\d+)_(\d+)', base)
        if not match:
            continue

        subj_id, rep_num, task_type, label_num, task_rep_num = match.groups()
        local_url = os.path.join(patient_path, file)

        task_type_val = 1 if task_type == 'M' else 0
        label_str = label_map.get(label_num, 'Unknown')
        label_encoded = label_encoding.get(label_str, -1)

        rows.append([
            srno,
            patient_num,
            local_url,
            int(rep_num),
            task_type_val,
            int(task_rep_num),
            label_str,
            label_encoded
        ])
        srno += 1

# -----------------------------
# STEP 5: SAVE EXCEL
# -----------------------------
columns = [
    'srno',
    'patient_number',
    'local_url',
    'repetition_number',
    'task_type',
    'task_repetition_number',
    'task_label',
    'task_label_encoded'
]

df = pd.DataFrame(rows, columns=columns)
df.to_excel(META_PATH, index=False)

print(f'Metadata written to {META_PATH}')


ModuleNotFoundError: No module named 'openpyxl'